---
title: "DRG Cleaning v2"

author: "Carlos Resurreccion"

date: "2024-10-21"

---


# !!! MAKE SURE YOU RESTART THE (JUPYTER) R KERNEL BEFORE PROCEEDING !!!


# Parameters

Change which year to process in
`~/drg-pipeline/data-cleaning/cache/year_to_load.txt`

Change main GLOBAL (i.e. across all scripts) parameters in 
`~/drg-pipeline/data-cleaning/00a-parameters.r`


Change seldom touched parameters in
`~/drg-pipeline/data-cleaning/r_scripts_v2/0.1.0.params_fpaths.R`

In [1]:
source("~/drg-pipeline/data-cleaning/00a-parameters.r")


Parallelization: TRUE 


# Libraries


In [2]:
# Update the grouper
system("git submodule update --init --recursive")

# List required packages
required_packages <- c(
  "data.table", # Fast data manipulation
  "here", # Simplifies file path management
  "tictoc", # Timing code execution
  "stringr", # String manipulation
  "stringi", # Unicode string processing
  "lubridate", # Date-time handling
  "profvis", # Profiling R code
  "hash", # Hashing utility
  "future", # Parallel processing
  "future.apply", # Parallelized apply functions
  "knitr", # Dynamic report generation
  "htmlwidgets", # Interactive HTML widgets
  "parallelly", # Advanced parallel computing
  "stringdist", # String distance calculations
  "parallel", # Base parallel computing
  "reticulate", # Interface to Python
  "bigrquery", # BigQuery client
  "jsonlite", # JSON parsing
  "googleCloudStorageR", # Google Cloud Storage access
  "haven", # Read/write Stata, SPSS, SAS files
  "fst", # Fast serialization
  "httr", # HTTP requests
  "ggplot2", # Data visualization
  "rmarkdown", # Dynamic markdown documents
  "digest", # Create cryptographic hashes
  "base64enc", # Base64 encoding/decoding
  "arrow", # Apache Arrow for fast data storage
  "tidyverse" # Collection of data science packages
)

github_packages <- c(
  "r-lib/styler" # Code formatting
)

# Installation commands (commented out, for reference)
invisible(lapply(
  required_packages, function(pkg) {
    if (!require(pkg, character.only = TRUE)) {
      install.packages(pkg)
    }
  }
))
invisible(lapply(
  github_packages, function(repo) {
    if (!require(basename(repo), character.only = TRUE)) {
      remotes::install_github(repo)
    }
  }
))

# Load packages (assumes they are already installed)
invisible(lapply(required_packages, library, character.only = TRUE))
invisible(lapply(basename(github_packages), library, character.only = TRUE))


Loading required package: data.table

Loading required package: here

here() starts at /home/resurreccion_cmc/drg-pipeline

Loading required package: tictoc


Attaching package: ‘tictoc’


The following object is masked from ‘package:data.table’:

    shift


Loading required package: stringr

Loading required package: stringi

Loading required package: lubridate


Attaching package: ‘lubridate’


The following objects are masked from ‘package:data.table’:

    hour, isoweek, mday, minute, month, quarter, second, wday, week,
    yday, year


The following objects are masked from ‘package:base’:

    date, intersect, setdiff, union


Loading required package: profvis

Loading required package: hash

hash-2.2.6.3 provided by Decision Patterns



Attaching package: ‘hash’


The following object is masked from ‘package:tictoc’:

    clear


The following object is masked from ‘package:data.table’:

    copy


Loading required package: future

Loading required package: future.apply

Loading

# R Scripts


In [3]:
year <- 2018

# Source each file sequentially
for (file in list.files(
  here::here("data-cleaning/r_scripts_v2"),
  pattern = "\\.R$", full.names = TRUE
)) {
  invisible(source(file))
}

message(year)


ℹ 2025-02-10 05:19:24.744163 > Setting client.id from options(googleAuthR.client_id)

All directories exist.


Total Rows via cached object: 11777674

Utilizing 4 cores (8 threads)


2018



# Data Cleaning Proper


# Load Mapping Data


In [4]:
# Enable caching and printing options for data mapping
to_use_cache <- TRUE # Set to TRUE to enable saving and loading of .rds files
to_print_mapping_data <- FALSE # Set to TRUE to print mapping data tables

# Helper function to load data from cache or query from BigQuery if not cached
load_or_query <- function(query, var_name) {
  rds_path <- here(cache_path, "mapping", paste0(var_name, ".rds"))
  if (to_use_cache && file.exists(rds_path)) {
    if (verbose_output) message("Loading ", var_name, " from cache...")
    # Load data from .rds file if cache exists
    return(readRDS(rds_path))
  } else {
    if (verbose_output) message("Querying ", var_name, " from BigQuery...")
    # Query data from BigQuery if not cached
    # Query execution function (BigQuery to data.table)
    dt <- query_bq_to_dt(query)
    saveRDS(dt, rds_path) # Save queried data to .rds cache file
    return(dt)
  }
}

# Helper function to print all rows of a data.table if
# to_print_mapping_data is enabled
if (to_print_mapping_data) {
  print_all <- function(dt, title) {
    cat("\n---", title, "---\n") # Print table title
    print(dt, nrow = Inf) # Print all rows of the data.table
  }
}

# 1. Query and load the grouper_v5.proc table
# This table contains procedure codes and attributes
# like description, classification, and site
proc_query <- paste0("SELECT * FROM ", gcp_proj, ".grouper_v5.proc")
proc <- load_or_query(proc_query, "proc")
proc[, CODE := as.character(CODE)] # Ensure the CODE column is of character type

# 2. Query and load the phic.acr_rvs_map table
# This table maps RVS codes to ICD-9-CM codes,
# used for healthcare billing purposes
rvs_icd9_query <- paste0(
  "SELECT * FROM ",
  gcp_proj, ".phic_libraries.acr_rvs_map"
)
rvs_icd9 <- load_or_query(rvs_icd9_query, "rvs_icd9")

# Convert RVS and ICD9CM columns to character type
# and adjust ICD9CM for multiplication
rvs_icd9 <- rvs_icd9[, .(
  rvs = as.character(rvs),
  icd9cm = as.character(as.numeric(icd9cm) * 100)
)]

# Merge the RVS-ICD9 mapping with the proc table for DRG classification
rvs_icd9 <- merge(
  rvs_icd9,
  proc[, .(CODE, DRGUSE)], # Select CODE and DRGUSE columns for merging
  by.x = "icd9cm", by.y = "CODE", all.x = TRUE
  # Merge on icd9cm and CODE columns
)

# Filter and annotate DRG-related codes, removing unnecessary DRGUSE column
rvs_icd9 <- rvs_icd9[, is_drg := !is.na(DRGUSE) & DRGUSE][
  !is.na(rvs) & !is.na(icd9cm), -"DRGUSE"
]

# 3. Query and load phic.acr_procedure table
# This table contains RVS codes, relative value units (RVUs),
# and descriptions for procedures
acr_rvs_query <- paste0(
  "SELECT * FROM ",
  gcp_proj, ".phic_libraries.acr_procedure"
)
acr_rvs <- load_or_query(acr_rvs_query, "acr_rvs")

# 4. Query and load grouper_v5.i10 table
# This table contains ICD-10 codes with DRG grouping data,
# including codes marked as "accepted" (ACCPDX = "Y")
i10_query <- paste0("SELECT * FROM ", gcp_proj, ".grouper_v5.i10")
tdrg_icd10 <- load_or_query(i10_query, "tdrg_icd10")
setkey(tdrg_icd10, "CODE") # Set the CODE column as key for efficient lookups

# Extract unique accepted ICD-10 codes for diagnosis
# (ACCPDX == "Y") and store in acc_pdx
acc_pdx <- unique(tdrg_icd10[ACCPDX == "Y", CODE])

# Create an environment for quick lookup of accepted diagnosis codes
acc_pdx_env <- new.env(hash = TRUE, parent = emptyenv())
for (code in acc_pdx) {
  # Assign each accepted code to the environment
  assign(code, TRUE, envir = acc_pdx_env)
}

# 5. Query and load icd.phl_icd10 table
# This table lists diseases and their corresponding
# ICD-10 codes specific to the Philippines
phl_icd10_query <- paste0("SELECT * FROM ", gcp_proj, ".icd.phl_icd10")
phl_icd10 <- load_or_query(phl_icd10_query, "phl_icd10")

# Filter and process neoplasm codes by extracting
# specific codes from complex ICD-10 notations
neoplasms_dt_actual <- as.data.table(phl_icd10[
  # Select rows with '/' in icd10, indicating neoplasm codes
  grepl("/", icd10), .(icd10)
  # Extract relevant part
][, icd10 := sapply(strsplit(icd10, ","), function(x) trimws(x[2]))])


# 6. Query and load grouper_v5.i10vx table
# This table contains an expanded version of ICD-10 codes with validation flags
i10vx_query <- paste0("SELECT * FROM ", gcp_proj, ".grouper_v5.i10vx")
i10vx <- load_or_query(i10vx_query, "i10vx")
setkey(i10vx, "code") # Set the code column as key for efficient lookup
acc_icd <- unique(i10vx[, code]) # Extract unique ICD codes from this table
acc_icd_set <- unique(acc_icd)

# 7. Query and load hci.temp_hci table
# This table lists healthcare institutions with details
# like ownership, category, and location
hci_query <- paste0("SELECT * FROM ", gcp_proj, ".hci.temp_hci")
hci <- load_or_query(hci_query, "hci")

# 8. Define global variables for use later in the script:
neoplasm_codes <- unique(neoplasms_dt_actual$icd10) # Unique neoplasm codes
covid_codes <- unique(covid_rvs) # Unique COVID-related codes
rvs_codes <- unique(acr_rvs$rvs) # Unique RVS codes

neoplasm_pattern <- paste0("(", paste(neoplasm_codes, collapse = "|"), ")")
covid_pattern <- paste0("(", paste(covid_codes, collapse = "|"), ")")
rvs_pattern <- paste0("(", paste(rvs_codes, collapse = "|"), ")")

phil_icds <- unique(gsub(
  "[^A-Za-z0-9]", "",
  phl_icd10[!grepl("/", icd10), icd10]
))
icd_codes <- unique(tdrg_icd10$CODE)

# Function to create an environment from a vector of unique values
create_env_from_vector <- function(vec) {
  env <- new.env(parent = emptyenv())
  list2env(setNames(as.list(rep(TRUE, length(vec))), vec), envir = env)
  return(env)
}

# 1. Create environment for proc table data if specific values are needed
# Here we assume proc$CODE is the field of interest
proc_env <- create_env_from_vector(proc$CODE)

# 2. Create environment for rvs_icd9 table data based on rvs and icd9cm
rvs_env <- create_env_from_vector(rvs_icd9$rvs)
icd9cm_env <- create_env_from_vector(rvs_icd9$icd9cm)

# 3. Environment for acr_rvs table (assuming rvs is the field of interest)
acr_rvs_env <- create_env_from_vector(acr_rvs$rvs)

# 4. Environment for accepted ICD-10 codes (from tdrg_icd10)
acc_pdx_env <- create_env_from_vector(acc_pdx)

# 5. Environment for phl_icd10 ICD-10 codes (e.g., neoplasm codes)
phl_icd10_env <- create_env_from_vector(phl_icd10$icd10)

# 6. Environment for expanded ICD-10 codes (i10vx)
acc_icd_env <- create_env_from_vector(i10vx$code)

# 7. Environment for hci table data if needed for specific fields (e.g., id_hci)
# Assuming hci$id_hci is the identifier of interest
hci_env <- create_env_from_vector(hci$id_hci)

# 8. Other specific environments for global variables
neoplasm_env <- create_env_from_vector(neoplasm_codes)
covid_env <- create_env_from_vector(covid_codes)
rvs_codes_env <- create_env_from_vector(rvs_codes)
phil_icds_env <- create_env_from_vector(phil_icds)
icd_codes_env <- create_env_from_vector(icd_codes)

# Combine COVID and neoplasm codes into a single environment
covid_neoplasm_codes <- unique(c(covid_codes, neoplasm_codes))
covid_neoplasm_env <- create_env_from_vector(covid_neoplasm_codes)

# Combine COVID, RVS, and neoplasm codes into a
# single environment for efficient lookup
covid_rvs_neoplasm_codes <- unique(c(covid_codes, rvs_codes, neoplasm_codes))
covid_rvs_neoplasm_env <- create_env_from_vector(covid_rvs_neoplasm_codes)


# Create a combined regular expression pattern to match COVID,
# RVS, and neoplasm codes in data processing
covid_rvs_neoplasm_pattern <- paste(
  c(covid_codes, rvs_codes, neoplasm_codes),
  collapse = "|"
)


# Data Cleaning Function


In [ ]:
# Define process_chunk
process_chunk <- function(
    chunk, yr_to_load = year, col_maps = column_mappings,
    known_vals = known_values, remap_master = col_remap_master,
    avail_cols = available_columns) {
  # Rename Columns
  setnames(chunk,
    old = avail_cols[avail_cols %in% names(col_maps)],
    new = sapply(
      avail_cols[avail_cols %in% names(col_maps)],
      function(col) col_maps[[col]]
    )
  )

  if (!"id_year" %in% colnames(chunk)) {
    chunk[, id_year := as.integer(yr_to_load)]
  }

  # Trim whitespace for id columns AND
  # Make a copy of clin_c1 and clin_c2 for later use
  chunk[, `:=`(
    id_series = trimws(id_series),
    id_pin = trimws(id_pin), c1 = clin_c1, c2 = clin_c2
  )]

  # Handle decimal seconds for 2022 and 2023 data
  chunk[, (c("time_adm", "time_dis")) := lapply(
    .SD,
    function(col) {
      ifelse(
        grepl("AM|PM", col),
        format(as.POSIXct(sub("\\.\\d+ ", " ", col),
          format = "%m/%d/%Y %I:%M:%S %p"
        ), "%H:%M"),
        col
      )
    }
  ), .SDcols = c("time_adm", "time_dis")]

  # Collapse icd rvs columns
  cols_to_extract <- grep("^(clin_icd\\d+|clin_rvs\\d+)$",
    names(chunk),
    value = TRUE
  )

  clin_icd_cols <- lapply(
    cols_to_extract[grepl("^clin_icd", cols_to_extract)],
    function(col) chunk[[col]]
  )
  clin_rvs_cols <- lapply(
    cols_to_extract[grepl("^clin_rvs", cols_to_extract)],
    function(col) chunk[[col]]
  )
  col_list <- list()
  if (!is.null(clin_icd_cols) && length(clin_icd_cols) > 0) {
    col_list$clin_icd <- collapse_clean_icd_rvs_cols(
      clin_icd_cols,
      is_icd = TRUE
    )
  }
  if (!is.null(clin_rvs_cols) && length(clin_rvs_cols) > 0) {
    col_list$clin_rvs <- collapse_clean_icd_rvs_cols(
      clin_rvs_cols,
      is_icd = FALSE
    )
  }
  chunk[, `:=`(
    clin_icd = col_list$clin_icd,
    clin_rvs = col_list$clin_rvs
  )]
  chunk[, (grep("^(clin_icd\\d+|clin_rvs\\d+)$",
    names(chunk),
    value = TRUE
  )) := NULL]

  # Clean c1 and c2 columns
  c1_result <- clean_column(chunk$c1)
  c2_result <- clean_column(chunk$c2)
  chunk[, `:=`(c1_orig = c1, c1 = c1_result$cleaned_col)]
  chunk[, `:=`(c2_orig = c2, c2 = c2_result$cleaned_col)]

  # Apply is_covid boolean
  is_covid_c1 <- c1_result$is_covid
  is_covid_c2 <- c2_result$is_covid
  chunk[, is_covid := (is_covid_c1 | is_covid_c2)]

  # prepare icds for mapping
  chunk[, `:=`(
    c1 = lapply(c1, prep_icd_for_mapping),
    c2 = lapply(c2, prep_icd_for_mapping)
  )]

  chunk[, clin_icd := lapply(seq_len(.N), function(i) {
    clin_icd_list <- c(manual_replacement(clin_icd[[i]]), c1[[i]], c2[[i]])
    return(flatten_then_check_null_na(clin_icd_list))
  })]

  # Replace empty with NA, then replace with character(0)
  chunk <- replace_na_or_empty(
    dt = replace_na_or_empty(
      dt = chunk, replace_with = "NA_character_"
    ),
    replace_with = "character(0)"
  )

  # Move rvs icd rvs codes to proper columns
  # Process c1 then c2
  c1_results <- append_copy_remove_icd_rvs(
    chunk$c1, chunk$clin_rvs, chunk$clin_icd
  )
  c2_results <- append_copy_remove_icd_rvs(
    chunk$c2, chunk$clin_rvs, chunk$clin_icd
  )
  chunk[, `:=`(
    clin_rvs = c1_results$clin_rvs,
    c1 = c1_results$col, clin_icd = c1_results$clin_icd
  )]
  chunk[, `:=`(
    clin_rvs = c2_results$clin_rvs,
    c2 = c2_results$col, clin_icd = c2_results$clin_icd
  )]

  # Replace empty with NA, then replace with character(0)
  chunk <- replace_na_or_empty(
    dt = replace_na_or_empty(
      dt = chunk, replace_with = "NA_character_"
    ),
    replace_with = "character(0)"
  )

  # List of column names to remap
  cols_to_remap <- c(
    "pat_type", "pat_memcat_parent",
    "pat_memcat_child", "clin_discharge", "claim_status"
  )

  # Apply remapping in a single step
  chunk[, (cols_to_remap) := lapply(
    .SD,
    remap_patient_data, remap_master
  ), .SDcols = cols_to_remap]

  # Map RVS codes
  chunk[, icd9_list := map_rvs_icd9(clin_rvs)]

  # Prepare codes for mapping
  chunk[, `:=`(
    c1 = lapply(
      c1,
      function(x) {
        if (is.null(x) || all(is.na(x))) {
          return(character(0))
        } else {
          return(x)
        }
      }
    ),
    c2 = lapply(
      c2,
      function(x) {
        if (is.null(x) || all(is.na(x))) {
          return(character(0))
        } else {
          return(x)
        }
      }
    )
  )]

  # Map ICD 10 codes
  chunk[, `:=`(
    c1 = map_icd10(c1),
    c2 = map_icd10(c2),
    clin_icd = map_icd10(clin_icd)
  )]

  # Replace na/empty with character(0)
  chunk <- replace_na_or_empty(dt = chunk, replace_with = "character(0)")

  # Find pdx
  pdx_inputs <- prep_pdx_inputs(
    chunk$c1, chunk$c2, chunk$clin_icd,
    acc_pdx, neoplasms_dt_actual, acr_rvs, covid_rvs
  )
  pdx_result <- find_pdx(
    pdx_inputs$c1, pdx_inputs$c2, pdx_inputs$clin_icd,
    global_seed
  )
  chunk[, c("pdx", "pdx_code") := .(pdx_result$pdx, pdx_result$pdx_code)]
  chunk[, c("c1", "c2", "clin_icd") := lapply(.SD, function(col) {
    lapply(seq_len(.N), function(i) {
      lst <- col[[i]]
      pdx_val <- pdx[i]
      if (!is.na(pdx_val)) lst <- setdiff(lst, pdx_val)
      return(as.character(lst))
    })
  }), .SDcols = c("c1", "c2", "clin_icd")]

  # Restore clin_c1 and clin_c2 AND Save rvs mappings to clin_proc
  chunk[, `:=`(
    clin_c1 = c1_orig, clin_c2 = c2_orig,
    clin_proc = icd9_list
  )][, `:=`(c1_orig = NULL, c2_orig = NULL, icd9_list = NULL)]

  # Save icd mappings to clin_sdx
  chunk[, clin_sdx := Map(
    function(pdx_var, sdx_var) {
      sdx_var[sdx_var != pdx_var]
    }, pdx, clin_icd
  )][, clin_icd := NULL]

  # Safeguard: If pat_bdate doesn't exist, set it to NA_Date_
  if (!"pat_bdate" %in% colnames(chunk)) chunk[, pat_bdate := NA_Date_]

  # Format date columns and check for dates before 1900-01-01
  date_cols <- c(
    "date_adm", "date_dis", "date_rec", "date_ref",
    "date_check", "pat_bdate", "date_ext"
  )

  chunk[, (date_cols) := lapply(.SD, function(x) {
    x <- as.Date(x, format = "%m/%d/%Y")
    x[x < as.Date("1900-01-01")] <- NA_Date_
    return(x)
  }), .SDcols = date_cols]

  # Format time columns
  chunk[, c("time_adm", "time_dis") := lapply(.SD, function(x) {
    as.character(ifelse(is.na(x), "00:00:00", paste0(x, ":00")))
  }), .SDcols = c("time_adm", "time_dis")]

  # Set time to UTC so that datetime timestamp matches up
  # with time_adm and time_dis for groupings
  chunk[, date_adm := as.POSIXct(paste(date_adm, time_adm),
    format = "%Y-%m-%d %H:%M:%S", tz = "UTC"
  )]
  chunk[, date_dis := as.POSIXct(paste(date_dis, time_dis),
    format = "%Y-%m-%d %H:%M:%S", tz = "UTC"
  )]

  # Format clin_outpatient and clin_emergency
  chunk[, c("clin_outpatient", "clin_emergency") := lapply(.SD, function(col) {
    as.logical(as.integer(col))
  }), .SDcols = c("clin_outpatient", "clin_emergency")]

  # Safeguard: If pat_bwt doesn't exist, set it to NA_real_
  if (!"pat_bwt" %in% colnames(chunk)) chunk[, pat_bwt := NA_real_]

  # Format numeric columns as numeric
  num_cols <- c(
    "pat_age", "pat_bwt", "clin_discharge", "claim_payout",
    "claim_charge", "id_year", "pdx_code"
  )
  chunk[, (num_cols) := lapply(.SD, as.numeric), .SDcols = num_cols]

  # Format integer columns as integer
  int_cols <- c("clin_discharge", "id_year", "pdx_code")
  chunk[, (int_cols) := lapply(.SD, as.integer), .SDcols = int_cols]

  # Format character columns as character
  char_cols <- c(
    "id_hcp", "pat_type", "clin_acc", "pat_rel", "pat_sex",
    "pat_memcat_parent", "pat_memcat_child", "claim_status", "pdx"
  )
  chunk[, (char_cols) := lapply(.SD, as.character), .SDcols = char_cols]
  char_cols <- names(chunk)[sapply(chunk, is.character)] # WHY IS THIS HERE?

  # Initialize pat_ageday to NA_integer_
  chunk[, pat_ageday := NA_integer_]

  # Subset clin_sdx to valid icd codes only
  chunk[, clin_sdx := lapply(clin_sdx, function(codes) {
    valid_codes <- codes[codes %chin% acc_icd_set]
    return(ifelse(length(valid_codes) > 0, valid_codes, NA_character_))
  })]
  chunk[, clin_sdx := lapply(clin_sdx, function(x) {
    if (is.null(x)) character(0) else unlist(x)
  })]

  # Replace with character(0)
  chunk <- replace_na_or_empty(
    dt = chunk, replace_with = "character(0)",
    additional_columns = c("c1", "c2", "pdx")
  )

  # Convert char cols to UTF-8, then replace empty with NA_character_
  chunk[, (char_cols) := lapply(.SD, function(col) {
    col <- iconv(col, from = "", to = "UTF-8")
    col[col %chin% c("None", "")] <- NA_character_
    return(col)
  }), .SDcols = char_cols]

  # Replace nan with NA_real_
  num_cols <- names(chunk)[sapply(chunk, is.numeric)]
  chunk[, (num_cols) := lapply(.SD, function(col) {
    col[is.nan(col)] <- NA_real_
    return(col)
  }), .SDcols = num_cols]

  # Split id_hcp then replace empty with character(0)
  chunk[, ("id_hcp") := lapply(.SD, function(x) {
    lapply(strsplit(x, "\\s*,\\s*|\\|\\||\\|"), function(y) {
      ifelse(length(y) == 0L || all(is.na(y)), character(0), y)
    })
  }), .SDcols = "id_hcp"]
  chunk[, ("id_hcp") := lapply(.SD, function(x) {
    lapply(x, function(y) {
      ifelse(is.null(y) || length(y) == 0L || all(is.na(y)), character(0), y)
    })
  }), .SDcols = "id_hcp"]

  # Rename pdx code columns
  setnames(chunk, c("pdx", "pdx_code"), c("clin_pdx", "clin_pdx_source"))

  # Set final column order
  setcolorder(chunk, c(
    "id_year", "id_series", "id_pin", "id_hci", "id_hcp", "date_adm",
    "time_adm", "date_dis", "time_dis", "date_rec", "date_ref",
    "date_check", "date_ext", "pat_type", "pat_rel", "pat_bdate", "pat_age",
    "pat_ageday", "pat_sex", "pat_bwt", "pat_memcat_parent",
    "pat_memcat_child", "claim_status", "claim_payout",
    "claim_charge", "is_covid", "clin_discharge", "clin_outpatient",
    "clin_emergency", "clin_acc", "clin_c1", "c1", "clin_c2", "c2",
    "clin_sdx", "clin_proc", "clin_rvs", "clin_pdx", "clin_pdx_source"
  ))

  # Set clin_discharge as integer
  chunk[, clin_discharge := as.integer(clin_discharge)]

  # Chop off excess clin_sdx and clin_proc from the
  # RIGHT hand side (i.e. least priority ones)
  chunk[, clin_sdx := lapply(clin_sdx, function(x) head(x, 12))]
  chunk[, clin_proc := lapply(clin_proc, function(x) head(x, 20))]

  # Set pat_age to 0 for specific cases
  chunk[
    !is.na(pat_bdate) & !is.na(date_adm),
    pat_age := floor(as.numeric(as.Date(date_adm) - pat_bdate) / 365.25)
  ]
  chunk[!is.na(pat_bdate) & !is.na(date_adm) & !is.na(pat_age) &
    pat_bdate > as.Date(date_adm), pat_bdate := NA_Date_]
  chunk[grepl("99432", c1) & !is.na(pat_age) & pat_age < 0 &
    pat_age >= -1, pat_age := 0]
  chunk[
    !is.na(pat_age) & pat_age > 0 & pat_age <= 124,
    pat_age := floor(pat_age)
  ]
  chunk[
    !is.na(pat_age) & (pat_age < 0 | pat_age > 124),
    pat_age := NA_integer_
  ]

  invisible(gc())
  return(chunk)
}


# Data Cleaning Loop

In [6]:
# Data Cleaning Pipeline for DRG Processing
# This script processes large datasets in parts, applying
# parallel processing for efficiency.
# It reads, chunks, processes, and consolidates data before
# saving intermediate and final outputs.

for (loop_part in 1:split_parts) {
  start_time <- Sys.time() # Record start time for processing

  # Step 1: Read the appropriate file
  read_in_dt <- read_appropriate_file(loop_part)

  # Step 2: Split the data into chunks for parallel processing
  chunk_size <- ceiling(nrow(read_in_dt) / nthreads)
  chunks <- split(read_in_dt, rep(1:nthreads,
    each = chunk_size,
    length.out = nrow(read_in_dt)
  ))

  # Step 3: Process chunks in parallel or sequentially
  cat(paste0("\rStart processing part  ", loop_part, " of ", split_parts))
  flush.console()
  parallel_results <-
    if (to_parallel) {
      mclapply(chunks, process_chunk, mc.cores = nthreads)
    } else if (!to_debug) {
      lapply(chunks, process_chunk)
    } else {
      list(process_chunk(chunks[[1]]))
    }

  # Consolidate processed chunks
  summarized_dt <- rbindlist(parallel_results)

  # Step 4: Save processed data if required
  if (to_write) {
    saveRDS(
      summarized_dt, here(chkpt_1_path, paste0(
        chkpt_1_prefix, year, suffix,
        "part_", sprintf("%02d", loop_part), "_of_", split_parts, ".rds"
      )),
      compress = TRUE
    )
  }

  # Step 5: Log processing time and update status
  processing_times[[loop_part]] <- as.numeric(difftime(Sys.time(),
    start_time,
    units = "secs"
  ))
  print_status_update(loop_part, split_parts, processing_times, "clean")

  # Cleanup memory
  rm(read_in_dt, summarized_dt)
  invisible(gc())
}

# Step 6: Combine all processed parts into a master data table
master_dt_list <- mclapply(1:split_parts,
  function(split_part) {
    read_part <- readRDS(
      here(chkpt_1_path, paste0(
        chkpt_1_prefix, year, suffix, "part_",
        sprintf("%02d", split_part), "_of_", split_parts, ".rds"
      ))
    )
    return(read_part)
  },
  mc.cores = nthreads
)

# Merge all parts into a single data table
message("Commencing rbindlist")
master_dt <- rbindlist(master_dt_list, fill = TRUE)
rm(master_dt_list)
invisible(gc())
message("Finished rbindlist")

# Step 7: Save final processed data
if (to_write) {
  message("Commencing saveRDS")
  saveRDS(master_dt, here(
    chkpt_2_path, paste0(
      chkpt_2_prefix, year, suffix, ".rds"
    )
  ), compress = TRUE)
  message("Finished saveRDS")
}

# Save a pre-final version of the master dataset
message(paste0(
  "Saving ",
  paste0(chkpt_2_prefix, year, suffix, "tmp", ".rds")
))
saveRDS(master_dt, here(
  chkpt_2_path,
  paste0(chkpt_2_prefix, year, suffix, "tmp", ".rds")
), compress = TRUE)
message(paste0(
  "Finished saving ",
  paste0(chkpt_2_prefix, year, suffix, "tmp", ".rds")
))


Finished cleaning 15 of 15 parts in 30s (ETA 0s)       

Commencing rbindlist

Finished rbindlist

Commencing saveRDS

Finished saveRDS

Saving chkpt_2_claims_2018_sampled_625_tmp.rds

Finished saving chkpt_2_claims_2018_sampled_625_tmp.rds



# Temp Output for Verification of Refactor

In [7]:
fwrite(
  readRDS(here(
    chkpt_2_path,
    paste0(chkpt_2_prefix, year, suffix, "tmp", ".rds")
  )),
  "~/drg-pipeline/data-cleaning/debug/test.csv"
)


# BQ Preparation

In [8]:
# Final preparations for BQ upload
# Load the dataset from the tmp chkpt
result <- readRDS(here(
  chkpt_2_path,
  paste0(chkpt_2_prefix, year, suffix, "tmp", ".rds")
))

# Add is_covid variable
# Identifies COVID-related claims by checking multiple clinical fields
result[, is_covid := {
  covid_found <- rep(FALSE, .N) # Initialize all rows as FALSE

  # Check each field sequentially, marking matches as TRUE
  not_found <- !covid_found
  # Check primary diagnosis
  covid_found[not_found] <- clin_c1[not_found] %chin% covid_rvs

  not_found <- !covid_found
  # Check secondary diagnosis
  covid_found[not_found] <- clin_c2[not_found] %chin% covid_rvs

  not_found <- !covid_found
  # Check coded diagnosis
  covid_found[not_found] <- c2[not_found] %chin% covid_rvs

  not_found <- !covid_found
  # Check additional coded diagnosis
  covid_found[not_found] <- c1[not_found] %chin% covid_rvs

  not_found <- !covid_found
  covid_found[not_found] <- sapply(
    clin_rvs[not_found],
    function(row) any(row %chin% covid_rvs)
  ) # Check procedure codes

  not_found <- !covid_found
  covid_found[not_found] <- sapply(
    clin_sdx[not_found],
    function(row) any(row %chin% covid_rvs)
  ) # Check supporting diagnoses

  not_found <- !covid_found
  covid_found[not_found] <- sapply(
    clin_proc[not_found],
    function(row) any(row %chin% covid_rvs)
  ) # Check performed procedures

  covid_found # Return logical vector of COVID matches
}]

# Subset the dataset for BQ
# Keep only relevant columns needed for BigQuery upload
result <- result[, .(
  # Identifiers
  id_series, id_pin, id_hci, id_hcp,
  # Date-related fields
  date_adm, date_dis, date_rec, date_ref, date_check,
  # Patient details
  pat_type, pat_rel, pat_age, pat_ageday, pat_sex,
  pat_bwt, pat_memcat_parent, pat_memcat_child,
  # Claim-related fields
  claim_status, claim_payout, claim_charge, is_covid,
  # Clinical classification
  clin_discharge, clin_outpatient, clin_emergency, clin_acc,
  # Clinical details
  clin_c1, clin_c2, clin_sdx, clin_proc, clin_pdx, clin_pdx_source
)]

# Save the processed dataset to a new chkpt before BQ upload
saveRDS(result, here(
  chkpt_2_path,
  paste0(chkpt_2_prefix, year, suffix, "final", ".rds")
))


# BQ Upload

In [9]:
# BQ upload
if (to_bq) {
  # Define BQ table name
  if (!to_sample) bq_table <- paste0("claims_", year)

  # Attempt to delete the table if it exists
  tryCatch(
    bq_table_delete(bq_table(gcp_proj, bq_dataset, bq_table)),
    error = function(e) {
      if (grepl("Not found", e, ignore.case = TRUE)) {
        message("Table does not exist, nothing to drop.")
      } else {
        stop(e)
      }
    }
  )

  # Create the BQ table if it does not exist
  tryCatch(
    bq_table_create(
      bq_table(gcp_proj, bq_dataset, bq_table),
      fields = fromJSON(here(
        "data-cleaning/r_scripts_v2",
        "bq_schema_cleaning.json"
      ), simplifyDataFrame = FALSE)
    ),
    error = function(e) {
      if (grepl("already exists", e, ignore.case = TRUE)) {
        message("Table already exists. Skipping creation and upload.")
      } else {
        stop(e)
      }
    }
  )

  if (to_write) {
    # Define chunk size for upload
    chunk_size <- 250000
    # Calculate number of chunks
    num_chunks <- ceiling(nrow(result) / chunk_size)

    for (i in seq_len(num_chunks)) {
      # Extract chunk
      chunk <- result[
        ((i - 1) * chunk_size + 1):min(i * chunk_size, nrow(result)),
      ]

      # Upload chunk to BQ
      bq_table_upload(
        bq_table(gcp_proj, bq_dataset, bq_table),
        values = chunk,
        write_disposition = if (i == 1) "WRITE_EMPTY" else "WRITE_APPEND"
      )
    }
  }
}
